# HGP: from two repetition checks to a distance certificate

**Question.** How do the HGP matrices give a [[13,1,3]] code, and why is the proposed weight-three operator logical rather than a stabilizer?

**Paper locations.** Section V, “HGP Codes,” Eq. (44), and its distance argument (`sec:HGP`, `eq:HGP`).

**Companion paper.** Yuan Yao, Guo Zhang, Yingli Yang, Yuanye Zhu, Xiaosi Xu, Xiao Yuan, and Ying Li, **“An introduction to qLDPC codes from a structural perspective.”** Submitted.

The paper numbers refer to the manuscript identified by `PAPER_SOURCE` in
[`paper_cases.py`](../qldpc_generator/paper_cases.py). Source labels are retained
to help locate the same material after renumbering. Equation numbers below are
local to this notebook; code outputs are computational results.

**Before running.** Follow the [installation instructions](../README.md#installation-and-execution),
select that Python environment, and run the cells in order. All matrix arithmetic
and ranks are over GF(2); matrix indices start at zero. For the coverage overview,
start with [06 — Paper validation](06-paper-validation.ipynb).

In [1]:
import numpy as np
from IPython.display import Markdown, display

def show_table(headers, rows):
    """Format results for reading; no mathematical calculations happen here."""
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

## 1. State the inputs and the qubit ordering

The input is the length-three repetition code. Its check matrix has full row rank,
and its kernel is generated by the nonzero word $c=(1,1,1)$:

$$
H=\begin{pmatrix}1&1&0\\0&1&1\end{pmatrix},\qquad
r=2,\quad n_c=3,\quad k=n_c-\operatorname{rank}_2H=1. \tag{1}
$$

We use $H_1=H_2=H$. The library follows paper Eq. (44): the first four qubits
are check–check pairs (CC), and the remaining nine are variable–variable pairs
(VV). Both returned matrices have checks as rows and qubits as columns:

$$
H_X=(I_r\otimes H^{\mathsf T}\;\;H\otimes I_{n_c}),\qquad
H_Z=(H^{\mathsf T}\otimes I_r\;\;I_{n_c}\otimes H). \tag{2}
$$

The following cell compares the constructor with these displayed blocks for
this one example, and prints the complete small matrices.

In [2]:
from qldpc_generator import hgp
from qldpc_generator.gf2 import rank, nullspace, in_rowspace

H = np.array([[1, 1, 0], [0, 1, 1]], dtype=np.uint8)
r, n_c = H.shape
assert rank(H) == r and n_c - r > 0
I_r = np.eye(r, dtype=np.uint8)
I_n = np.eye(n_c, dtype=np.uint8)
code_hgp = hgp(H, H)
assert np.array_equal(code_hgp.hx, np.hstack([np.kron(I_r, H.T), np.kron(H, I_n)]))
assert np.array_equal(code_hgp.hz, np.hstack([np.kron(H.T, I_r), np.kron(I_n, H)]))
print("H_X (CC columns 0–3; VV columns 4–12):\n", code_hgp.hx)
print("H_Z (same column order):\n", code_hgp.hz)

H_X (CC columns 0–3; VV columns 4–12):
 [[1 0 0 0 1 0 0 1 0 0 0 0 0]
 [1 1 0 0 0 1 0 0 1 0 0 0 0]
 [0 1 0 0 0 0 1 0 0 1 0 0 0]
 [0 0 1 0 0 0 0 1 0 0 1 0 0]
 [0 0 1 1 0 0 0 0 1 0 0 1 0]
 [0 0 0 1 0 0 0 0 0 1 0 0 1]]
H_Z (same column order):
 [[1 0 0 0 1 1 0 0 0 0 0 0 0]
 [0 1 0 0 0 1 1 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 1 1 0 0 0 0]
 [0 1 0 1 0 0 0 0 1 1 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 1 1 0]
 [0 0 0 1 0 0 0 0 0 0 0 1 1]]


The two cross terms of $H_XH_Z^{\mathsf T}$ cancel in characteristic two.
The transpose input code is zero-dimensional because $H$ has full row rank.
The predicted parameters and the independently computed binary ranks agree:

$$
N=n_c^2+r^2=13,\qquad K=k^2=1
   =N-\operatorname{rank}_2H_X-\operatorname{rank}_2H_Z. \tag{3}
$$

In [3]:
assert not np.any(code_hgp.hx @ code_hgp.hz.T % 2)
p = code_hgp.parameters()
assert (p['n'], p['k'], p['rank_x'], p['rank_z']) == (13, 1, 6, 6)
show_table(["Quantity", "Computed result"], [
    ["H_X / H_Z shapes", f"{code_hgp.hx.shape} / {code_hgp.hz.shape}"],
    ["N / K", f"{p['n']} / {p['k']}"],
    ["GF(2) ranks of H_X / H_Z", f"{p['rank_x']} / {p['rank_z']}"],
    ["Nonzero entries in H_X H_Z^T", 0],
])

| Quantity | Computed result |
| --- | --- |
| H_X / H_Z shapes | (6, 13) / (6, 13) |
| N / K | 13 / 1 |
| GF(2) ranks of H_X / H_Z | 6 / 6 |
| Nonzero entries in H_X H_Z^T | 0 |

## 2. Make the quotient argument visible

An operator with zero syndrome need not be logical: it might be a stabilizer.
The paper separates these possibilities using the quotient by $\operatorname{rowsp}H$.
Here that quotient can be represented in coordinates by a one-row matrix $Q$:

$$
Q=\begin{pmatrix}1&1&1\end{pmatrix},\qquad
\ker Q=\operatorname{rowsp}H,\qquad QH^{\mathsf T}=0. \tag{4}
$$

Choose $e_0=(1,0,0)$, so $Qe_0=1$. This explicitly verifies that $e_0$ is
outside the row space. The candidate and the map on its VV block are

$$
z=(0_{r^2},c\otimes e_0),\qquad
\Pi=I_{n_c}\otimes Q,\qquad
\Pi(c\otimes e_0)=c\otimes(Qe_0)\ne0. \tag{5}
$$

Every Z stabilizer has a VV block in the image of $I_{n_c}\otimes H^{\mathsf T}$,
which $\Pi$ annihilates. Therefore a candidate with a nonzero image under $\Pi$
cannot be a Z stabilizer. Full row rank and $k>0$ are the hypotheses of the
symmetric certificate helper used below; they were checked in Section 1.

In [4]:
c = np.array([1, 1, 1], dtype=np.uint8)
e0 = np.array([1, 0, 0], dtype=np.uint8)
Q = nullspace(H)  # Rows span ker(H); hence ker(Q) = rowspace(H).
assert Q.tolist() == [[1, 1, 1]]
assert not np.any(Q @ H.T % 2)
assert rank(Q) == n_c - rank(H)
assert not in_rowspace(e0, H)
Pi = np.kron(I_n, Q)
z = np.r_[np.zeros(r*r, dtype=np.uint8), np.kron(c, e0)]
stabilizer_image = Pi @ code_hgp.hz[:, r*r:].T % 2
candidate_image = Pi @ z[r*r:] % 2
assert not np.any(code_hgp.hx @ z % 2)
assert not np.any(stabilizer_image) and np.any(candidate_image)
assert code_hgp.is_logical(z, 'Z')
show_table(["Object or check", "Result"], [
    ["c / e_0 / Q", f"{c.tolist()} / {e0.tolist()} / {Q.tolist()}"],
    ["H_X z", (code_hgp.hx @ z % 2).tolist()],
    ["Pi applied to every Z-stabilizer VV block", "all zero"],
    ["Pi applied to the candidate VV block", candidate_image.tolist()],
    ["Nonzero qubit indices in z", np.flatnonzero(z).tolist()],
    ["Weight of z", int(z.sum())],
])
from qldpc_generator.certificates import hgp_distance_witness
certificate = hgp_distance_witness(H)
assert np.array_equal(certificate['logical'], z)
assert certificate['upper_bound'] == 3

| Object or check | Result |
| --- | --- |
| c / e_0 / Q | [1, 1, 1] / [1, 0, 0] / [[1, 1, 1]] |
| H_X z | [0, 0, 0, 0, 0, 0] |
| Pi applied to every Z-stabilizer VV block | all zero |
| Pi applied to the candidate VV block | [1, 1, 1] |
| Nonzero qubit indices in z | [4, 7, 10] |
| Weight of z | 3 |

The support indices 4, 7, and 10 correspond to VV pairs (0,0), (1,0), and (2,0).
The first factor follows the classical word; the second stays at one coordinate.

### Failure control: zero syndrome alone is insufficient

Replace the chosen coordinate vector by the first row of H. The new operator
still has zero syndrome, but is now a Z stabilizer. This is exactly the distinction
that the quotient argument must make.

In [5]:
bad = np.r_[np.zeros(r*r, dtype=np.uint8), np.kron(c, H[0])]
assert not np.any(code_hgp.hx @ bad % 2)
assert not np.any(Pi @ bad[r*r:] % 2)
assert in_rowspace(bad, code_hgp.hz)
assert not code_hgp.is_logical(bad, 'Z')
show_table(["Candidate", "Zero syndrome?", "Z stabilizer?", "Nontrivial logical?"], [
    ["c tensor e_0", True, False, True],
    ["c tensor H[0]", True, True, False],
])

| Candidate | Zero syndrome? | Z stabilizer? | Nontrivial logical? |
| --- | --- | --- | --- |
| c tensor e_0 | True | False | True |
| c tensor H[0] | True | True | False |

## 3. Separate the upper bound from exact distance

The constructed logical operator proves $D\le3$. To certify equality for this
finite example, the search must also exclude all nontrivial X and Z logical
operators of weights one and two:

$$
D\ge3\quad\text{(exhaustive exclusion below weight 3)},\qquad
D\le3\quad\text{(explicit witness)},\qquad D=3. \tag{6}
$$

Stabilizers are excluded by the search. The two calls below intentionally show
the difference between a lower bound and an exact result.

In [6]:
lower = code_hgp.distance(max_weight=2)
exact = code_hgp.distance(max_weight=3)
assert lower['status'] == 'bounded' and lower['lower_bound'] == 3
assert exact['status'] == 'exact' and exact['value'] == 3
show_table(["Search", "Status", "Conclusion"], [
    ["All supports through weight 2", lower['status'], "D >= 3"],
    ["All supports through weight 3", exact['status'], "D = 3"],
])
show_table(["Sector", "Minimum weight", "Logical support (zero-based)"], [
    [sector, data['value'], data['witness']] for sector, data in exact['sectors'].items()
])
LX, LZ = code_hgp.logical_operators()
assert np.array_equal(LX @ LZ.T % 2, np.eye(code_hgp.k, dtype=np.uint8))
print("Logical pairing L_X L_Z^T =", (LX @ LZ.T % 2).tolist())

| Search | Status | Conclusion |
| --- | --- | --- |
| All supports through weight 2 | bounded | D >= 3 |
| All supports through weight 3 | exact | D = 3 |

| Sector | Minimum weight | Logical support (zero-based) |
| --- | --- | --- |
| X | 3 | [4, 5, 6] |
| Z | 3 | [4, 7, 10] |

Logical pairing L_X L_Z^T = [[1]]


## 4. What this establishes

This example verifies the defining matrices, the quotient certificate, and the
exact finite distance of [[13,1,3]]. The general HGP lower bound remains the
theorem cited in Section V; this finite search does not establish asymptotic
scaling for an arbitrary classical family.

Implementation: [construction](../qldpc_generator/constructions.py),
[certificate](../qldpc_generator/certificates.py), and
[tests of products and hypotheses](../tests/test_products.py).
Continue with [02 — LP and LCS](02-lp-lcs.ipynb).